# Replicating Article Figures and Tables

This notebook reproduces all figures and tables reported in the paper directly
from pre-computed benchmark results.

## Using Pre-Computed Results

The `results/article/` folder containing all benchmark results is committed to
the git repository. No experiments need to be rerun. The results are already
available and can be used directly to replicate every figure and table.

## Contents

| Item | Description |
|---|---|
| **Table 1** | Compression results (bits per sample by dataset) |
| **Table 2** | Compression speed (MB/s per machine, mean ± std, with PDZ speedup ratios) |
| **Table 3** | Decompression speed (MB/s per machine, mean ± std, with PDZ speedup ratios) |
| **Figure 3** | Compression ratio vs. speed trade-off |
| **Supplementary Figure 1** | Size and time split decomposition |
| **Supplementary Figure 2** | Peak memory usage vs. file size |

In [ ]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")

plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["font.size"] = 10
pd.set_option("display.max_columns", 60)

sns.set_theme(style="whitegrid")

In [ ]:
bootstrap_candidate = Path.cwd().resolve()
for current in [bootstrap_candidate, *bootstrap_candidate.parents]:
    if (current / "results").is_dir() and (current / "scripts" / "benchmarks").is_dir():
        REPO_ROOT = current
        break
else:
    raise FileNotFoundError(
        "Could not locate the repository root from the current working directory."
    )

SCRIPTS_BENCHMARKS = REPO_ROOT / "scripts" / "benchmarks"
if str(SCRIPTS_BENCHMARKS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_BENCHMARKS))

from analysis.compression import (
    DEFAULT_COMPRESSION_LATEX_MACRO_LABELS,
    build_compression_artifacts,
    build_compression_run_entries,
    build_subject_last_compression_latex_table,
    build_subject_last_compression_report_table,
    build_subject_vs_baseline_compression_prd_frame,
)
from analysis.memory import build_memory_artifacts, plot_memory_peak_vs_file_size
from analysis.shared import CANONICAL_ALGORITHM_ORDER, detect_repo_root
from analysis.splits import build_split_artifacts, plot_split_decomposition
from analysis.speed import (
    build_speed_artifacts,
    configure_speed_runs,
    plot_tradeoff_bits_per_sample_vs_speed,
    summarize_machine_metric,
)

REPO_ROOT = detect_repo_root(REPO_ROOT)
ARTICLE_RESULTS_OUTPUT_DIR = REPO_ROOT / "notebooks" / "outputs" / "article_results"
ARTICLE_RESULTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTICLE_RESULTS_ROOT = REPO_ROOT / "results" / "article"

ARTICLE_ALGORITHMS = ["VBZ", "PDZ", "EX-ZD-ZSTD"]
ROUND_DIGITS = 4

print(f"Repository root: {REPO_ROOT}")
print(f"Article results root: {ARTICLE_RESULTS_ROOT}")
print(f"Notebook output directory: {ARTICLE_RESULTS_OUTPUT_DIR}")
print("Canonical algorithm order:", ", ".join(CANONICAL_ALGORITHM_ORDER))
print("Article comparison order:", ", ".join(ARTICLE_ALGORITHMS))

In [ ]:
EXPORT_FIGURES = True
EXPORT_FIGURE_FORMAT = "pdf"
EXPORT_INCLUDE_TITLES_IN_EXPORTED_FIGURES = False
FIGURE_EXPORT_KWARGS = {
    "export_enabled": EXPORT_FIGURES,
    "output_dir": ARTICLE_RESULTS_OUTPUT_DIR,
    "export_format": EXPORT_FIGURE_FORMAT,
    "include_titles": EXPORT_INCLUDE_TITLES_IN_EXPORTED_FIGURES,
}

print(f"Figure export enabled: {EXPORT_FIGURES}")
print(f"Figure export format: {EXPORT_FIGURE_FORMAT}")
print(
    "Titles included in exported figures:",
    EXPORT_INCLUDE_TITLES_IN_EXPORTED_FIGURES,
)

## Tables

### Table 1: Compression Results

Bits per sample by dataset for VBZ, EX-ZD-ZSTD, and PDZ, together with PDZ
percent relative difference (PRD) versus each baseline.
A negative PRD means PDZ achieves a lower bits-per-sample value (better
compression).

In [ ]:
COMPRESSION_RESULTS_ROOT = ARTICLE_RESULTS_ROOT / "compression"
COMPRESSION_DATASET_RUN_IDS = {
    "DS1": "20260430T202949_VBZ-EX-ZD-ZSTD-PDZ_full_DS1",
    "DS2": "20260430T204946_VBZ-EX-ZD-ZSTD-PDZ_full_DS2",
    "DS3": "20260430T210335_VBZ-EX-ZD-ZSTD-PDZ_full_DS3",
    "DS4": "20260430T212502_VBZ-EX-ZD-ZSTD-PDZ_full_DS4",
    "DS5": "20260430T213949_VBZ-EX-ZD-ZSTD-PDZ_full_DS5",
    "DS6-RNA": "20260430T215044_VBZ-EX-ZD-ZSTD-PDZ_full_DS6",
    "DS7": "20260430T221118_VBZ-EX-ZD-ZSTD-PDZ_full_DS7",
    "DS8": "20260430T221836_VBZ-EX-ZD-ZSTD-PDZ_full_DS8",
    "DS9": "20260430T221947_VBZ-EX-ZD-ZSTD-PDZ_full_DS9",
    "DS10-RNA": "20260430T222410_VBZ-EX-ZD-ZSTD-PDZ_full_DS10",
}
COMPRESSION_REPORT_SUBJECT_ALGORITHM = "PDZ"
COMPRESSION_PRD_BASELINE_ALGORITHMS = ["VBZ", "EX-ZD-ZSTD"]
COMPRESSION_ARTICLE_ALGORITHMS = [
    *COMPRESSION_PRD_BASELINE_ALGORITHMS,
    COMPRESSION_REPORT_SUBJECT_ALGORITHM,
]
COMPRESSION_LATEX_MACRO_LABELS = dict(DEFAULT_COMPRESSION_LATEX_MACRO_LABELS)
COMPRESSION_LATEX_OUTPUT_PATH = (
    ARTICLE_RESULTS_OUTPUT_DIR / "compression" / "compression_ratio_table.tex"
)
COMPRESSION_LATEX_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

COMPRESSION_RUNS = build_compression_run_entries(
    COMPRESSION_RESULTS_ROOT,
    COMPRESSION_DATASET_RUN_IDS,
)

print(f"Compression results root: {COMPRESSION_RESULTS_ROOT}")
print(f"Configured compression runs: {len(COMPRESSION_RUNS)}")
print(f"Compression LaTeX output path: {COMPRESSION_LATEX_OUTPUT_PATH}")

In [ ]:
compression_runs, compression_algorithms, compression_df = build_compression_artifacts(
    COMPRESSION_RUNS,
    requested_algorithms=COMPRESSION_ARTICLE_ALGORITHMS,
)
compression_dataset_order = [run["label"] for run in compression_runs]

compression_table = compression_df[
    [
        "dataset",
        "algorithm",
        "bits_per_sample",
        "file_count",
        "total_chunks",
        "total_samples",
        "total_compressed_bytes",
        "successful_chunk_count",
        "failed_chunk_count",
        "dataset_success_rate",
        "summary_status",
    ]
].copy()
compression_table["bits_per_sample"] = pd.to_numeric(
    compression_table["bits_per_sample"], errors="coerce"
).round(ROUND_DIGITS)
compression_table["dataset_success_rate"] = pd.to_numeric(
    compression_table["dataset_success_rate"], errors="coerce"
).round(ROUND_DIGITS)

compression_bps_matrix = compression_table.pivot(
    index="dataset",
    columns="algorithm",
    values="bits_per_sample",
).reindex(index=compression_dataset_order, columns=compression_algorithms)
compression_prd_frame = build_subject_vs_baseline_compression_prd_frame(
    compression_bps_matrix,
    subject_algorithm=COMPRESSION_REPORT_SUBJECT_ALGORITHM,
    baseline_algorithms=COMPRESSION_PRD_BASELINE_ALGORITHMS,
)
compression_report_table = build_subject_last_compression_report_table(
    compression_prd_frame,
    compression_dataset_order,
    subject_algorithm=COMPRESSION_REPORT_SUBJECT_ALGORITHM,
    baseline_algorithms=COMPRESSION_PRD_BASELINE_ALGORITHMS,
)
compression_latex_table = build_subject_last_compression_latex_table(
    compression_prd_frame,
    compression_dataset_order,
    subject_algorithm=COMPRESSION_REPORT_SUBJECT_ALGORITHM,
    baseline_algorithms=COMPRESSION_PRD_BASELINE_ALGORITHMS,
    latex_macro_labels=COMPRESSION_LATEX_MACRO_LABELS,
)
COMPRESSION_LATEX_OUTPUT_PATH.write_text(compression_latex_table, encoding="utf-8")

print("Table 1: Compression Results")
print(
    f"Non-{COMPRESSION_REPORT_SUBJECT_ALGORITHM} entries show "
    f"{COMPRESSION_REPORT_SUBJECT_ALGORITHM}-relative PRD: "
    f"({COMPRESSION_REPORT_SUBJECT_ALGORITHM} - baseline) * 100 / baseline."
)
display(compression_report_table)
print(f"LaTeX written to: {COMPRESSION_LATEX_OUTPUT_PATH}")

### Table 2: Compression Speed

Mean ± std compression speed in MB/s per machine, with PDZ speedup ratios
relative to VBZ and EX-ZD-ZSTD.

In [ ]:
SPEED_RESULTS_ROOT = ARTICLE_RESULTS_ROOT / "speed"
SPEED_MACHINE_LABELS: dict[str, str] | None = {
    "x86S1": "x86 S1",
    "x86S2": "x86 S2",
    "x86D1": "x86 D1",
    "x86D2": "x86 D2",
    "x86L1": "x86 L1",
    "x86L2": "x86 L2",
    "arm64L1": "arm64 L1",
    "arm64L2": "arm64 L2",
}
SPEED_MACHINE_DENYLIST: list[str] = []

SPEED_RUNS = configure_speed_runs(
    SPEED_RESULTS_ROOT,
    machine_labels=SPEED_MACHINE_LABELS,
    denylist=SPEED_MACHINE_DENYLIST,
)

print(f"Speed results root: {SPEED_RESULTS_ROOT}")
print("Configured speed runs:")
for entry in SPEED_RUNS:
    machine_name = str(entry.get("machine_name", entry["label"]))
    display_name = str(entry["label"])
    if machine_name == display_name:
        print(f"  - {display_name}")
    else:
        print(f"  - {machine_name} -> {display_name}")

In [ ]:
speed_context = build_speed_artifacts(
    SPEED_RUNS,
    requested_algorithms=ARTICLE_ALGORITHMS,
)
speed_algorithms = speed_context["algorithms"]
speed_machine_order = speed_context["machine_order"]
speed_comparison_df = speed_context["comparison_df"]

print("Speed algorithms:", ", ".join(speed_algorithms))
print("Machine order:", ", ".join(speed_machine_order))

In [ ]:
SPEED_LATEX_MACRO_LABELS = {
    "PDZ": r"\algShort",
    "VBZ": "VBZ",
    "EX-ZD-ZSTD": r"\exzd",
}
SPEED_LATEX_SUBJECT = "PDZ"
SPEED_LATEX_BASELINES = ["VBZ", "EX-ZD-ZSTD"]
SPEED_LATEX_DISPLAY_ALGORITHMS = ["VBZ", "EX-ZD-ZSTD", "PDZ"]
SPEED_LATEX_OUTPUT_DIR = ARTICLE_RESULTS_OUTPUT_DIR / "speed"
SPEED_LATEX_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _build_speed_latex_table_single(
    summary: pd.DataFrame,
    machine_order: list[str],
    algorithms: list[str],
    subject: str,
    baselines: list[str],
    macro_labels: dict[str, str],
    stage: str,
    latex_label: str,
) -> str:
    mean_piv = summary.pivot_table(
        index="machine", columns="algorithm", values="Mean", aggfunc="first"
    ).reindex(index=machine_order, columns=algorithms)
    std_piv = summary.pivot_table(
        index="machine", columns="algorithm", values="Std", aggfunc="first"
    ).reindex(index=machine_order, columns=algorithms)

    def _fmt_speed(mean, std) -> str:
        if pd.isna(mean):
            return "---"
        if pd.isna(std):
            return f"{mean:.1f}"
        return f"{mean:.1f} $\\pm$ {std:.1f}"

    def _fmt_ratio(subj_mean, base_mean) -> str:
        if pd.isna(subj_mean) or pd.isna(base_mean) or base_mean == 0:
            return "---"
        return f"{subj_mean / base_mean:.2f}$\\times$"

    subject_label = macro_labels.get(subject, subject)
    baseline_labels = [macro_labels.get(b, b) for b in baselines]
    algo_labels = [macro_labels.get(a, a) for a in algorithms]

    n_algo_cols = len(algorithms)
    n_ratio_cols = len(baselines)
    col_spec = "|l|" + "r|" * n_algo_cols + "r|" * n_ratio_cols

    algo_header = " & ".join(algo_labels)
    ratio_header = " & ".join(f"{subject_label}/{bl}" for bl in baseline_labels)

    lines = [
        r"\begin{table}[]",
        r"\centering",
        (
            f"\\caption{{{stage.capitalize()} speed (MB/s, mean~$\\pm$~std) for"
            f" VBZ, {macro_labels.get('EX-ZD-ZSTD', 'EX-ZD-ZSTD')}, and {subject_label}"
            f" per machine, with {subject_label} speed-up ratios relative to each baseline.}}"
        ),
        f"\\begin{{tabular}}{{{col_spec}}}",
        r"\hline",
        f"Machine & {algo_header} & {ratio_header}" + r"\\",
        r"\hline",
    ]

    for machine in machine_order:
        cells = []
        for algo in algorithms:
            cells.append(
                _fmt_speed(mean_piv.at[machine, algo], std_piv.at[machine, algo])
            )
        subj_mean = mean_piv.at[machine, subject]
        for base in baselines:
            cells.append(_fmt_ratio(subj_mean, mean_piv.at[machine, base]))
        lines.append(f"{machine} & " + " & ".join(cells) + r"\\")

    lines += [
        r"\hline",
        r"\end{tabular}",
        f"\\label{{{latex_label}}}",
        r"\end{table}",
    ]
    return "\n".join(lines)


def _build_speed_display_table(
    summary: pd.DataFrame,
    machine_order: list[str],
    algorithms: list[str],
    subject: str,
    baselines: list[str],
) -> pd.DataFrame:
    """Build a human-readable DataFrame for notebook display."""
    mean_piv = summary.pivot_table(
        index="machine", columns="algorithm", values="Mean", aggfunc="first"
    ).reindex(index=machine_order, columns=algorithms)
    std_piv = summary.pivot_table(
        index="machine", columns="algorithm", values="Std", aggfunc="first"
    ).reindex(index=machine_order, columns=algorithms)

    rows = {}
    for algo in algorithms:
        col = []
        for machine in machine_order:
            mean = mean_piv.at[machine, algo]
            std = std_piv.at[machine, algo]
            if pd.isna(mean):
                col.append("---")
            elif pd.isna(std):
                col.append(f"{mean:.1f}")
            else:
                col.append(f"{mean:.1f} \u00b1 {std:.1f}")
        rows[f"{algo} (MB/s)"] = col

    for base in baselines:
        col = []
        for machine in machine_order:
            subj_mean = mean_piv.at[machine, subject]
            base_mean = mean_piv.at[machine, base]
            if pd.isna(subj_mean) or pd.isna(base_mean) or base_mean == 0:
                col.append("---")
            else:
                col.append(f"{subj_mean / base_mean:.2f}\u00d7")
        rows[f"{subject}/{base}"] = col

    return pd.DataFrame(rows, index=pd.Index(machine_order, name="Machine"))


comp_summary = summarize_machine_metric(speed_comparison_df, "compression")
comp_latex = _build_speed_latex_table_single(
    comp_summary,
    speed_machine_order,
    SPEED_LATEX_DISPLAY_ALGORITHMS,
    SPEED_LATEX_SUBJECT,
    SPEED_LATEX_BASELINES,
    SPEED_LATEX_MACRO_LABELS,
    stage="compression",
    latex_label="tab:speed_comp",
)
(SPEED_LATEX_OUTPUT_DIR / "speed_compression_table.tex").write_text(
    comp_latex, encoding="utf-8"
)
print("Table 2: Compression Speed")
display(
    _build_speed_display_table(
        comp_summary,
        speed_machine_order,
        SPEED_LATEX_DISPLAY_ALGORITHMS,
        SPEED_LATEX_SUBJECT,
        SPEED_LATEX_BASELINES,
    )
)
print(f"LaTeX written to: {SPEED_LATEX_OUTPUT_DIR / 'speed_compression_table.tex'}")

### Table 3: Decompression Speed

Mean ± std decompression speed in MB/s per machine, with PDZ speedup ratios
relative to VBZ and EX-ZD-ZSTD.

In [ ]:
decomp_summary = summarize_machine_metric(speed_comparison_df, "decompression")
decomp_latex = _build_speed_latex_table_single(
    decomp_summary,
    speed_machine_order,
    SPEED_LATEX_DISPLAY_ALGORITHMS,
    SPEED_LATEX_SUBJECT,
    SPEED_LATEX_BASELINES,
    SPEED_LATEX_MACRO_LABELS,
    stage="decompression",
    latex_label="tab:speed_decomp",
)
(SPEED_LATEX_OUTPUT_DIR / "speed_decompression_table.tex").write_text(
    decomp_latex, encoding="utf-8"
)
print("Table 3: Decompression Speed")
display(
    _build_speed_display_table(
        decomp_summary,
        speed_machine_order,
        SPEED_LATEX_DISPLAY_ALGORITHMS,
        SPEED_LATEX_SUBJECT,
        SPEED_LATEX_BASELINES,
    )
)
print(f"LaTeX written to: {SPEED_LATEX_OUTPUT_DIR / 'speed_decompression_table.tex'}")

## Figures

### Figure 3: Bits per Sample vs. Speed Trade-off

Each point represents one algorithm on one machine. The x-axis shows mean bits
per sample and the y-axis shows mean speed (MB/s). Circles mark compression,
crosses mark decompression, and the dashed segment links both measurements for
the same algorithm. Algorithms are annotated directly in each panel; points
closer to the upper-left corner combine better compression and higher speed.

In [ ]:
speed_tradeoff_df = plot_tradeoff_bits_per_sample_vs_speed(
    speed_comparison_df,
    speed_algorithms,
    speed_machine_order,
    **FIGURE_EXPORT_KWARGS,
)
display(speed_tradeoff_df.round(4))

### Supplementary Figure 1: Splits

Left panel: where the bytes go inside a compressed POD5 file (compressed signal
vs. signal table overhead vs. remaining file).
Right panel: where the time goes during compression and decompression
(signal processing vs. other overhead).

In [ ]:
SIZE_SPLIT_RESULTS_ROOT = ARTICLE_RESULTS_ROOT / "size_split"
TIME_SPLIT_RESULTS_ROOT = ARTICLE_RESULTS_ROOT / "time_split"

split_context = build_split_artifacts(
    SIZE_SPLIT_RESULTS_ROOT,
    TIME_SPLIT_RESULTS_ROOT,
)
split_overall_df = split_context["overall_df"].copy()

print("Size-split source run:", split_context["size_run_dir"])
print("Time-split source run:", split_context["time_run_dir"])

plot_split_decomposition(split_overall_df, **FIGURE_EXPORT_KWARGS)

### Supplementary Figure 2: Memory

Peak RSS vs. uncompressed file size for VBZ and PDZ. Circles = compression,
squares = decompression. Both axes are logarithmic. Lines are least-squares
linear fits in log-log space, summarising memory scaling behaviour with file
size.

In [ ]:
MEMORY_RESULTS_ROOT = ARTICLE_RESULTS_ROOT / "memory"

memory_context = build_memory_artifacts(MEMORY_RESULTS_ROOT)
memory_points_df = memory_context["points_df"].copy()

plot_memory_peak_vs_file_size(memory_points_df, **FIGURE_EXPORT_KWARGS)